# 03 - Silver to Gold Transformation

**Project:** Gender-Based Sales KPI Dashboard  
**Author:** Hicham ERRIHANI  
**Layer:** Silver → Gold  

This notebook aggregates sales data by gender, year, and category to build the Gold layer.

In [ ]:
from pyspark.sql.functions import col, sum as _sum, count, year

STORAGE_ACCOUNT = "stgendersaleshicham"
SILVER_PATH = f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
GOLD_PATH = f"abfss://gold@{STORAGE_ACCOUNT}.dfs.core.windows.net/"

In [ ]:
# Read Silver tables
dim_customer = spark.read.parquet(f"{SILVER_PATH}DimCustomer/")
fact_sales = spark.read.parquet(f"{SILVER_PATH}FactInternetSales/")

# Join
sales_with_customer = fact_sales.join(dim_customer, on="CustomerKey", how="inner")
print(f"Joined rows: {sales_with_customer.count()}")

In [ ]:
# Aggregate: Sales by Gender and Year
sales_by_gender_year = sales_with_customer \
    .withColumn("OrderYear", year(col("OrderDate"))) \
    .groupBy("Gender", "OrderYear") \
    .agg(
        _sum("SalesAmount").alias("TotalSales"),
        count("SalesOrderNumber").alias("OrderCount")
    ) \
    .orderBy("OrderYear", "Gender")

sales_by_gender_year.show(20)

In [ ]:
# Aggregate: Sales by Gender, Year and Product Category
sales_by_gender_category = sales_with_customer \
    .withColumn("OrderYear", year(col("OrderDate"))) \
    .groupBy("Gender", "OrderYear", "EnglishProductCategoryName") \
    .agg(_sum("SalesAmount").alias("TotalSales")) \
    .orderBy("OrderYear", "Gender", "EnglishProductCategoryName")

sales_by_gender_category.show(20)

In [ ]:
# Write to Gold layer
sales_by_gender_year.write.mode("overwrite").parquet(f"{GOLD_PATH}sales_by_gender_year/")
sales_by_gender_category.write.mode("overwrite").parquet(f"{GOLD_PATH}sales_by_gender_category/")

print("Gold layer written successfully.")